# S6E11 · Inspect → profile → shift check → CV baseline → submission, with `tabular-ds-mcp`

This notebook runs the tools of [tabular-ds-mcp](https://github.com/YusufErem/tabular-ds-mcp), an open-source MCP server that lets AI coding agents (Claude Code, Cursor, Claude Desktop) inspect and validate tabular data. Here the same functions are called directly from Python, so you can see exactly what an agent would get back.

Principles: every score is shown next to a no-skill baseline, nothing is fitted on rows it is scored on, and the validation scheme follows the data (random, time-ordered or grouped).

In [ ]:
# Internet must be ON (Settings → Internet). Installs the package from GitHub.
!pip install -q git+https://github.com/YusufErem/tabular-ds-mcp.git

In [ ]:
import json, os
import pandas as pd
import tabular_ds_mcp as t

DATA = "/kaggle/input/playground-series-s6e11"
if not os.path.exists(DATA):            # run locally on the synthetic demo data
    DATA = "demo_data"
show = lambda d, keys=None: print(json.dumps({k: d[k] for k in (keys or d) if k in d}, indent=1, ensure_ascii=False))

## 0 · What kind of problem is this?
`inspect_competition` reads the folder: target and ID (from the sample submission), task, submission format, a metric hint, time or group structure, extra tables. Its `suggested_calls` carry the right arguments for the next steps.

In [ ]:
plan = t.inspect_competition(DATA)
show(plan, ["target", "id_col", "task", "submission_format", "metric_hint", "time_columns", "group_columns",
            "support", "settings", "limitations", "notes"])
TRAIN, TEST, SAMPLE = plan["files"]["train"], plan["files"]["test"], plan["files"]["sample_submission"]
TARGET, ID = plan["target"], plan["id_col"]
SPLIT = {k: v for k, v in plan["settings"].items() if k in ("time_col", "group_col")}
METRIC = plan["settings"].get("metric", "roc_auc")   # S6E11 is scored with ROC-AUC (Evaluation page)
print(TARGET, ID, SPLIT, METRIC)

## 1 · Profile the training data
Rows, types, missing values, target balance and warnings (constant or ID-like columns, possible leakage, dates, repeated keys).

In [ ]:
profile = t.profile_dataset(TRAIN, target=TARGET, id_col=ID)
show(profile, ["rows", "n_features", "duplicate_rows", "target", "warnings"])
pd.DataFrame(profile["column_summary"]).set_index("column")

## 2 · Does the test set look like the training set?
PSI and KS per feature, plus adversarial validation: a model tries to tell train rows from test rows. AUC ≈ 0.5 means it cannot, so CV scores should transfer. The tool also checks whether test rows lie later in time or belong to unseen keys.

In [ ]:
shift = t.compare_train_test(TRAIN, TEST, target=TARGET, id_col=ID, **SPLIT)
show(shift, ["adversarial_auc", "shift_verdict", "adversarial_auc_without_time", "time_checks", "group_checks", "warnings"])
pd.DataFrame(shift["features_by_psi"]).head(10)

## 3 · Cross-validated baseline with the competition metric, next to a no-skill baseline
Gradient boosting; encoders are fitted inside each fold. With a rare positive class, read PR-AUC against the positive rate. If the data were time-ordered or grouped, `SPLIT` would switch the folds and the output would say how much random folds overstate the score.

In [ ]:
cv = t.cv_baseline(TRAIN, target=TARGET, id_col=ID, metric=METRIC, **SPLIT)
show(cv, ["validation", "metric", "cv_scores", "fold_spread", "no_skill_baseline", "random_split_comparison", "warnings"])

## 4 · Which features matter?
Permutation importance on a 20% hold-out: the drop in ROC-AUC when a feature is shuffled.

In [ ]:
expl = t.explain_model(TRAIN, target=TARGET, id_col=ID, **SPLIT)
show(expl, ["holdout", "holdout_scores", "features_with_no_gain"])
pd.DataFrame(expl["top_features"])

## 5 · Fit on all rows, write the submission in the sample's format, check it before uploading

In [ ]:
show(t.fit_predict(TRAIN, TEST, target=TARGET, id_col=ID, out_path="submission.csv",
                   sample_submission_path=SAMPLE, metric=METRIC))
check = t.check_submission("submission.csv", SAMPLE, test_path=TEST)
print("ready to upload:", check["ok"])
pd.DataFrame(check["checks"])

## Use the same tools from an AI agent
```bash
pip install git+https://github.com/YusufErem/tabular-ds-mcp.git
claude mcp add tabular-ds -- tabular-ds-mcp
```
Then ask Claude Code: *“Inspect the data folder, run the checks it suggests and give me a baseline with the competition metric.”* The repository also ships a skill (`skills/tabular-playbook`) that tells the agent the order of the checks, when to switch to time or group folds, and how to report them.